In [ ]:
# capstone project -- function 8 (8D), week N

import numpy as np
import warnings
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.spatial import ConvexHull
from scipy.stats import spearmanr

from bayes_tools import (
    normalize, initial_bounds, validate_bounds_consistency,
    generate_next_point, ucb_acquisition,
    append_observations, compute_iteration_diagnostics,
    fit_gp, get_length_scales, loo_predictions,
    compare_kappa_proposals, print_kappa_comparison,
    backtest_acquisitions, print_backtest_summary,
)
from viz_tools import (
    plot_nd_slices, plot_loo_calibration, plot_kappa_sensitivity,
    plot_acquisition_backtest,
    plot_convergence, plot_acquisition_decay, plot_uncertainty_shrinkage,
    plot_step_distance,
)

# Function 8

8D -- the highest-dimensional function in the set -- with 40 initial points plus 1 collected. **Using UCB with `kappa=0.25`, searching the six informative axes** (x0, x1, x2, x3, x5, x6) with x4 and x7 held at the incumbent.

Proposal this week: **`[0.035760, 0.0, 0.075310, 0.0, 0.586936, 0.318288, 0.0, 0.603135]`** (GP mean +10.155, std 0.202; the current best is 9.92799).

There is no project-wide best approach; each function is decided from its own evidence.

## A note on the recorded y value

`y = 9.9279892899925`, read from `outputs.txt`. It was supplied in conversation as "2 9.9279892899925", which looks like an editor gutter artifact -- the file itself contains no leading 2, and `9.928` is a plausible modest improvement on the previous best of `9.598` where `29.928` would be a 3x jump out of nowhere. **If 29.93 was actually intended, the analysis below changes substantially** and the notebook should be re-run: a value that far out would dominate the kernel fit the way function 4's -215 did.

## This is the best-behaved model in the set

Three independent signs, all pointing the same way:

- **Out-of-sample calibration is excellent.** Fitted on the initial 40 points, the GP predicted the new point at `9.8172 +/- 0.0727` against an actual `9.9280` -- a **1.5 sigma** miss. Function 5's equivalent was 17 sigma, function 4's LOO miss on its outlier was worse still.
- **LOO coverage is 40/41** inside 95% intervals, and the worst-predicted point is an ordinary initial observation missed by a small margin -- not the newly collected one, and not the incumbent.
- **The GP and the raw data agree on which axes matter.** x0, x2 and x6 have both the largest GP mean-variation and the strongest rank correlations. On function 5 these two diagnostics flatly contradicted each other; here they line up.

The collected point is also an **interpolation** -- every coordinate inside the range already sampled on its own axis, none on a bound -- and it is the new best, by `+0.330` on a spread of about 4.34. Clean provenance too: no sign flip (contrast functions 4 and 7) and no rounding overshoot (contrast function 4).

## What 8D breaks

Two tools stop being useful at this dimensionality, and it is worth being explicit rather than reading their output naively.

**The convex-hull test is vacuous.** All 41 observations are hull vertices, and the hull encloses **0.41%** of the observed bounding box. With 41 points in 8D there is essentially no interior, so every conceivable proposal reads as "outside the hull". The notebook computes it for continuity with functions 3-7 but the per-axis and bound checks carry all the weight. (`Delaunay` triangulation is also skipped here -- it is expensive and uninformative at this dimensionality.)

**`compute_iteration_diagnostics` needs `domain_grid_n` lowered by a lot.** Its `domain_mean_std` field averages the posterior std over a `domain_grid_n ** D` grid, and the default of 40 means `40**8 = 6.6e12` points -- around **419,000 GB**. See that cell; the notebook sizes it to `4`.

## Almost every acquisition wants a box corner, and that is a dimensionality effect

Searching all eight axes, the proposals pin coordinates to bounds almost immediately:

| config | coords on a bound | coords outside observed range | pred. mean |
|---|---|---|---|
| `exploit` | 3 of 8 | 3 | +10.74 |
| `ucb` kappa=0.5 | 5 of 8 | 6 | +10.74 |
| `ucb` kappa=1 | 6 of 8 | 7 | +10.70 |
| `ucb` kappa=2 | **8 of 8** | 8 | +10.61 |
| `ucb` kappa=5 | 8 of 8 | 8 | +7.73 |
| `max_variance` | 8 of 8 | 8 | -2.35 |

With `pad_fraction=1.0` doubling all eight axes, the box has roughly 2^8 = 256 corners and almost all of its volume is far from any data, so anything with a variance term runs outward. Even pure `exploit` pins three coordinates.

**EI and PI are outright degenerate here**, as on function 4: both propose a region with predicted mean **+0.018** against an incumbent of 9.93, and `xi` makes no difference (1% and 5% of the spread give identical answers). The improvement term is hopeless across the whole domain, so EI collapses onto its `sigma*phi(z)` term and becomes a variance-seeker in disguise. UCB is the only family here with a working dial.

## The fix: hold the near-irrelevant axes, and distinguish the two kinds of bound

**x4 and x7 are flat** -- x7's length-scale is pinned at the 10.0 ceiling and x4's sits at 9.71, and sweeping them moves the mean by 0.86 and 0.083 against x2's 7.30. Their rank correlations are also negligible (-0.098, +0.061). Holding them at the incumbent removes all upper-bound contact:

| `kappa` | pred. mean | on **upper** bound (artifact) | on lower bound (x >= 0 floor) | dist |
|---|---|---|---|---|
| 0 (= `exploit`) | +10.156 | none | x3 | 0.524 |
| **0.25** | **+10.155** | **none** | **x1, x3, x6** | **0.537** |
| 0.5 | +10.154 | none | x1, x3, x6 | 0.547 |
| 1.0 | +10.144 | none | x0, x1, x3, x6 | 0.584 |

The two bound types are not equivalent. **Upper** bounds come from `pad_fraction` and are arbitrary -- a coordinate pinned there means the padding is choosing it, not the model. **Lower** bounds are all `0.0`, a real domain constraint (X is never negative), so a coordinate there is a genuine "as low as physically allowed" prediction. That reading is supported independently: x0, x2 and x6 all have clearly negative rank correlations with `y` (-0.631, -0.602, -0.331), so low values on those axes really do look better.

Holding x4 costs some predicted mean -- the unrestricted `exploit` proposal reaches +10.74 by pushing x4 to its upper bound, against +10.16 here. That trade is deliberate: about 0.58 of predicted mean, which is model extrapolation, in exchange for removing an arbitrary excursion on an axis the model says barely matters.

`kappa=0.25` is chosen over `0` for a small exploration hedge at negligible cost (+10.155 vs +10.156), and over `1.0` because that starts pulling x0 to the floor as well.

## What the backtest says

| config | mean regret | median regret |
|---|---|---|
| `exploit` | 0.038 | 0 |
| `ucb_k1` | 0.054 | 0 |
| `ucb_k0.5` | 0.058 | 0 |
| `ei` | 0.061 | 0 |
| `pi` | 0.066 | 0 |
| `ucb_k2` | 0.095 | 0 |
| `ucb_k5` | 0.190 | 0 |
| `max_var` | 2.138 | 2.485 |

Strongly favours exploitation, and with 41 points the splits are more informative than on the smaller functions -- but the caveat still applies. The metric scores recognition of points whose `y` is already known, which rewards ranking by posterior mean and gives no credit for reducing uncertainty, so it favours `exploit` and low `kappa` on every function regardless (it ranked function 3's chosen config last). Note seven of eight configs achieve a median regret of **0**: the discrimination is almost entirely in the tail, where `max_variance` is 50x worse than the field. Read it as agreeing with the choice, not establishing it.

## No y-scaling

`y` spans 5.592 to 9.928, so a default `xi=0.01` is 0.23% of the spread -- inside the 0.1%-10% band where scaling is unnecessary. `kappa` is dimensionless regardless; the backtest's PI/EI rows use `xi` sized at 1% of the spread.

## Dimensionality-specific choices

- D=8, so `plot_2d_bo` doesn't apply -- the GP/acquisition view is **`plot_nd_slices`**, which gives eight panels. Expect x7 to be a flat line and x4 nearly so.
- `plot_bo_diagnostics` and `plot_sample_trajectory` hard-code a 2D scatter panel, so they're **skipped**; the individual trend plots are dimension-agnostic and used instead.
- Convex hull reported but vacuous; `Delaunay` skipped entirely. See above.

In [ ]:
X_initial = np.load("initial_data/function_8/initial_inputs.npy")
y_initial = np.load("initial_data/function_8/initial_outputs.npy")
n_initial = len(y_initial)
D = X_initial.shape[1]

assert D == 8, f"Expected an 8D problem, got {D}D input -- check the loaded file."

# Once, at the very start of the capstone:
new_X = np.empty((0, D))
new_y = np.empty((0,))

## Update this once per week

Include the new X and y values from the previous week, oldest first -- row order must be true chronological order, or `compute_iteration_diagnostics` at the bottom is meaningless.

One observation is recorded, and it is the new best. Provenance is clean here: all coordinates positive, none on a bound, all inside the range already sampled on their own axis -- no sign flip (contrast functions 4 and 7) and no rounding overshoot (contrast function 4).

**One thing to double-check**: `y` is recorded as `9.9279892899925`, matching `outputs.txt`. It was supplied as "2 9.9279892899925", which appears to be an editor artifact. If the intended value was `29.9279892899925`, change it below and re-run everything -- a value that far above the rest would dominate the kernel fit the way function 4's -215 does, and none of the conclusions in the header would survive unchanged.

In [ ]:
# Append last week's result BEFORE proposing this week's point, e.g.:
#
# new_X, new_y = append_observations(new_X, new_y, x_next, the_result_you_got)

new_X = np.array([
    # week 2
    [0.095333, 0.261314, 0.113603, 0.282728, 0.586936, 0.54168, 0.291424, 0.603135],
    # week 3 -- x1/x3/x6 landed exactly on the 0.0 floor.
    [0.035759, 0.0, 0.07531, 0.0, 0.586936, 0.318282, 0.0, 0.603135],
])

new_y = np.array([
    9.9279892899925,    # from outputs.txt. See the note above if 29.93 was intended.
    9.8020526511435,
])

print("observations collected so far:", len(new_y))
print("week 2 y: %.6f | week 3 y: %.6f (change %+.6f)"
      % (new_y[0], new_y[-1], new_y[-1] - new_y[0]))
print("best before week 3: %.6f | best overall now: %.6f"
      % (max(y_initial.max(), new_y[0]), max(y_initial.max(), new_y.max())))
print("-> week 3 is slightly BELOW week 2: three coordinates were driven to the")
print("   0.0 floor and the response did not improve, so that direction is spent")


## Build the full dataset (initial + everything collected so far)

In [ ]:
X, y = append_observations(X_initial, y_initial, new_X, new_y)

print("X_initial shape:", X_initial.shape, "| combined X shape:", X.shape)
print("y range: %.6f to %.6f (spread %.6f, std %.6f)"
      % (y.min(), y.max(), y.max() - y.min(), y.std()))

# X is known to never be negative -- lower_limit=0.0 is mandatory. Note this
# makes the LOWER bound a real domain constraint while the upper bounds are an
# artifact of pad_fraction; the checks below treat them differently.
bounds = initial_bounds(X_initial, pad_fraction=1.0, lower_limit=0.0)
validate_bounds_consistency(X, bounds)
print("\nBounds (upper):", np.round(bounds[:, 1], 4))

# The 0.0 floor is a real domain constraint, so a coordinate sitting exactly
# on it is legitimate -- but it also means the axis has run out of room in
# that direction, which is worth seeing per week.
print("\ncollected points vs the INITIAL batch's range:")
for r in range(len(new_X)):
    n_out = n_floor = 0
    for d in range(D):
        outside = (new_X[r, d] > X_initial[:, d].max()
                   or new_X[r, d] < X_initial[:, d].min())
        n_out += outside
        n_floor += new_X[r, d] == 0.0
        print(f"  week {r + 2}  x{d}={new_X[r, d]:.6f}  initial range"
              f" [{X_initial[:, d].min():.4f}, {X_initial[:, d].max():.4f}]"
              f"  outside={outside}{'  <- on the 0.0 floor' if new_X[r, d] == 0.0 else ''}")
    print(f"  -> week {r + 2}: {n_out} of {D} coordinates outside,"
          f" {n_floor} on the 0.0 floor, y={new_y[r]:.6f}")

# At D=8 the convex hull is vacuous: with this few points there is essentially
# no interior, so every proposal reads as outside it. Quantified rather than
# assumed. Delaunay is skipped -- expensive and uninformative at this D.
try:
    ch = ConvexHull(X)
    box_volume = float(np.prod(X.max(axis=0) - X.min(axis=0)))
    print(f"\nobserved box volume {box_volume:.4g} | convex hull {ch.volume:.4g}"
          f" ({ch.volume / box_volume:.2%} of it)")
    print(f"{len(ch.vertices)} of {len(X)} observations are hull vertices")
    if len(ch.vertices) == len(X):
        print("-> EVERY point is a vertex: the hull has no interior, so a")
        print("   hull-membership test carries no information at this D.")
except Exception as exc:
    print("\nconvex hull unavailable:", exc)

xi_frac = 0.01 / (y.max() - y.min())
print(f"\nxi=0.01 is {xi_frac:.4%} of the y spread"
      f" -> {'scaling NOT needed' if 0.001 <= xi_frac <= 0.1 else 'CONSIDER y-scaling'}")
print("(kappa is dimensionless, so UCB is unaffected either way)")

## Is the model stable and calibrated?

First, refit on the initial batch alone and predict the collected point -- a genuine out-of-sample test, since that fit never saw it. The miss in units of its own predicted sigma says how far the model's uncertainty can be trusted. This came back at 17 sigma on function 5 and changed the entire approach there; expect about 1.5 sigma here, the best in the set.

Second, compare kernels before and after. A single observation reshaping the model is a reason to distrust variance-driven acquisitions -- function 4's length-scales tripled from one point. Here the notable change is x3 coming *unpinned* (8.41 to 3.63), i.e. the new point made an axis look more relevant rather than less.

In [ ]:
gp_before = fit_gp(X_initial, y_initial, bounds, n_restarts_optimizer=25, random_state=0)
gp_check = fit_gp(X, y, bounds, n_restarts_optimizer=25, random_state=0)

# Fit is the initial batch only, so every collected row is out of sample. The
# model that actually proposed week 3 also had week 2, so this understates
# what was known at proposal time for later rows.
mu_b, sd_b = gp_before.predict(normalize(new_X, bounds), return_std=True)
print("out-of-sample test on the collected points (fit = initial batch only):")
for r in range(len(new_X)):
    miss = new_y[r] - mu_b[r]
    print(f"  week {r + 2}: predicted {mu_b[r]:+.6f} +/- {sd_b[r]:.6f}"
          f"   actual {new_y[r]:+.6f}")
    print(f"    miss {miss:+.6f} = {miss / sd_b[r]:+.1f} sigma"
          f"  -> {'well calibrated' if abs(miss / sd_b[r]) < 3 else 'POORLY calibrated'}")

ls_before, ls_after = get_length_scales(gp_before), get_length_scales(gp_check)
print("\nkernel before:", gp_before.kernel_)
print("kernel after :", gp_check.kernel_)
print("\nlength-scales before:", np.round(ls_before, 3))
print("length-scales after :", np.round(ls_after, 3))
print("ratio (after/before):", np.round(ls_after / ls_before, 3))

PINNED = 10.0  # fit_gp's default length_scale_bounds upper limit
print("\npinned (GP treats as irrelevant) before:",
      [d for d, v in enumerate(ls_before) if v >= 0.999 * PINNED] or "none")
print("pinned after                          :",
      [d for d, v in enumerate(ls_after) if v >= 0.999 * PINNED] or "none")

moved = np.max(np.abs(ls_after / ls_before - 1))
print(f"\nlargest relative length-scale change: {moved:.0%}")
if moved > 0.5:
    print("*** A length-scale moved by more than 50% from one observation. Check")
    print("    WHICH one: an axis becoming MORE relevant (length-scale shrinking)")
    print("    is benign; the dangerous case is an amplitude/length-scale blow-up")
    print("    that inflates far-field variance, as on function 4. ***")

## Which axes matter

Length-scales alone aren't sufficient: pinned at the 10.0 ceiling means "smooth, nearly linear over this domain", not "no effect" -- a mild monotone trend can still be present, and if it is, pushing along that axis is a real prediction rather than an optimiser artifact. That was function 5's trap.

So this cell measures GP mean-variation directly and pairs it with model-free rank correlations, then applies the flat-axis rule: flat if mean-variation is under 5% of the largest **or** the length-scale is at/above 9.0.

**This is the one function where the two diagnostics agree.** Expect x0, x2 and x6 to top both the mean-variation and the (absolute) rank-correlation ranking, and x4/x7 to come last on both. On function 5 they contradicted each other outright, which is why that notebook has to hedge; here they corroborate.

Note x4 qualifies as flat on the length-scale test (9.71) rather than the mean-variation test (0.86, which is 12% of x2's 7.30). Including it in the search is what pushed the unrestricted proposals onto x4's upper bound, so the length-scale criterion is doing real work.

In [ ]:
incumbent = X[np.argmax(y)]
print("incumbent (best observed):", np.round(incumbent, 6))
print("y = %.6f" % y.max())

print("\n%3s | %17s | %15s | %10s | %9s | %s"
      % ("ax", "GP mean-variation", "same, observed", "len-scale", "spearman", "rank agree?"))
spans = np.zeros(D)
srs = np.zeros(D)
for d in range(D):
    row = []
    for lo, hi in [(bounds[d, 0], bounds[d, 1]), (X[:, d].min(), X[:, d].max())]:
        grid = np.tile(incumbent, (300, 1))
        grid[:, d] = np.linspace(lo, hi, 300)
        m, _ = gp_check.predict(normalize(grid, bounds), return_std=True)
        row.append(m.max() - m.min())
    spans[d] = row[0]
    srs[d] = spearmanr(X[:, d], y)[0]

rank_span = np.argsort(np.argsort(-spans))
rank_sr = np.argsort(np.argsort(-np.abs(srs)))
for d in range(D):
    agree = "yes" if abs(int(rank_span[d]) - int(rank_sr[d])) <= 1 else "no"
    print("%3d | %17.4g | %15.4g | %10.3f | %+9.3f | %s (%d vs %d)"
          % (d, spans[d], 0.0 if d < 0 else spans[d], ls_after[d], srs[d],
             agree, rank_span[d] + 1, rank_sr[d] + 1))

FLAT_FRAC = 0.05
flat = [d for d in range(D)
        if spans[d] < FLAT_FRAC * spans.max() or ls_after[d] >= 0.9 * PINNED]
live = [d for d in range(D) if d not in flat]
print(f"\nflat axes (mean-variation < {FLAT_FRAC:.0%} of max, or length-scale >= {0.9 * PINNED}):",
      flat or "none")
for d in flat:
    why = []
    if spans[d] < FLAT_FRAC * spans.max():
        why.append("low mean-variation")
    if ls_after[d] >= 0.9 * PINNED:
        why.append(f"length-scale {ls_after[d]:.2f}")
    print(f"  x{d}: {', '.join(why)}")
print("axes searched:", live)
order = np.argsort(spans)[::-1]
print("dominance: x%d is only %.1fx the next (x%d) -> no single dominant axis"
      % (order[0], spans[order[0]] / spans[order[1]], order[1]))

## Backtest acquisition functions (using only data already collected)

Repeatedly splits the data into a "seed" set (fits the GP) and a held-out "candidate" set (true y known, hidden from the fit), then sees which config would have picked the best candidate most often. No new evaluations spent. `xi` is sized as 1% of the `y` spread.

With 41 points this is better powered than on the smaller functions -- but the structural caveat is unchanged: the metric scores recognition of points whose `y` is already known, which rewards ranking by posterior mean and gives no credit for reducing uncertainty, so it favours `exploit` and low `kappa` on every function regardless of what is appropriate. It ranked function 3's chosen config last.

Watch how little discrimination there is among the top seven: expect all of them at **median regret 0**, with the separation almost entirely in `max_variance`'s tail. Read it as agreeing with the choice rather than establishing it.

In [ ]:
xi_raw = 0.01 * (y.max() - y.min())
backtest_configs = [
    {"name": "ucb_k0.5", "acquisition": "ucb", "kappa": 0.5},
    {"name": "ucb_k1",   "acquisition": "ucb", "kappa": 1.0},
    {"name": "ucb_k2",   "acquisition": "ucb", "kappa": 2.0},
    {"name": "ucb_k5",   "acquisition": "ucb", "kappa": 5.0},
    {"name": "ei",       "acquisition": "ei",  "xi": xi_raw},
    {"name": "pi",       "acquisition": "pi",  "xi": xi_raw},
    {"name": "exploit",  "acquisition": "exploit"},
    {"name": "max_var",  "acquisition": "max_variance"},
]
print(f"xi for the PI/EI rows: {xi_raw:.6f} (1% of the y spread)\n")

backtest_results = backtest_acquisitions(
    X, y, bounds, backtest_configs,
    n_repeats=50, seed_frac=0.5, maximize=True,
    gp_kwargs={"n_restarts_optimizer": 15}, random_state=0,
)
print_backtest_summary(backtest_results)

fig = plot_acquisition_backtest(backtest_results)
plt.show()

## Why not EI or PI: they are degenerate on this function

Worth demonstrating rather than asserting, because the backtest above ranks them respectably -- that is the recognition metric talking, not their actual proposals.

Run at any `xi`, EI and PI here propose a region with predicted mean around **+0.018** against an incumbent of 9.93. The mechanism is the one seen on function 4: the improvement term `mu - y_best - xi` is hopeless everywhere in this large 8D box, so EI collapses onto its `sigma*phi(z)` term and silently becomes a variance-seeker. Sweeping `xi` across two orders of magnitude changes nothing, which is the tell -- if `xi` is not a live dial, the acquisition is not doing what its name says.

UCB is the only family here whose exploration weight remains controllable.

In [ ]:
spread = y.max() - y.min()
print(f"{'config':>16} | {'pred mean':>10} | {'pred std':>9} | on bound | outside obs")
for acq, xi in [("ei", 0.01 * spread), ("ei", 0.05 * spread), ("ei", 0.25 * spread),
                ("pi", 0.01 * spread), ("pi", 0.25 * spread)]:
    xn, g = generate_next_point(X, y, bounds, acquisition=acq, xi=xi, maximize=True,
                                 n_restarts=40, random_state=0)
    m, s = g.predict(normalize(xn.reshape(1, -1), bounds), return_std=True)
    edge = sum(np.isclose(xn[d], bounds[d, 0]) or np.isclose(xn[d], bounds[d, 1])
               for d in range(D))
    out = sum(not (X[:, d].min() <= xn[d] <= X[:, d].max()) for d in range(D))
    print(f"{acq + ' xi=' + format(xi, '.3f'):>16} | {m[0]:+10.4f} | {s[0]:9.4f}"
          f" | {edge:>3}/{D}   | {out}/{D}")
print(f"\nincumbent y = {y.max():.4f} -- these proposals are catastrophically worse,")
print("and xi has no effect across a 25x range, so it is not a live dial.")
print("EI/PI have degenerated into variance-seekers. Use UCB.")

## Compare kappa values -- the deciding cell

Two views. First `compare_kappa_proposals`, unrestricted over all eight axes, shown to demonstrate the corner problem: with `pad_fraction=1.0` doubling every axis, the box has around 256 corners and almost all its volume sits far from any data, so anything carrying a variance term runs outward. Expect coordinate-on-bound counts to climb from 3 of 8 at `kappa=0` to 8 of 8 by `kappa=2`.

Then the restricted search over the live axes only, holding x4 and x7 at the incumbent, which is what the choice is made from.

Bound contact is split into two categories because they mean different things. **Upper** bounds come from `pad_fraction` and are arbitrary -- a coordinate pinned there means the padding is choosing it rather than the model, and that is disqualifying. **Lower** bounds are all `0.0`, a real domain constraint, so a coordinate there is a genuine "as low as physically allowed" prediction -- and on x0, x2 and x6 that is corroborated by clearly negative rank correlations.

`kappa=0.25` is committed: it touches no upper bound, costs essentially nothing against pure exploitation (+10.155 vs +10.156), and keeps a small hedge. `kappa >= 1` starts pulling x0 to the floor as well.

In [ ]:
def bound_report(p):
    """Upper-bound contact is a pad_fraction artifact; lower is the X >= 0 floor."""
    upper = [d for d in range(D) if np.isclose(p[d], bounds[d, 1])]
    lower = [d for d in range(D) if np.isclose(p[d], bounds[d, 0])]
    return upper, lower


print("=== unrestricted over all 8 axes (shown to demonstrate the corner problem) ===")
kappa_rows = compare_kappa_proposals(X, y, bounds, kappa_values=[0.0, 0.5, 1.0, 2.0, 5.0],
                                      maximize=True, n_restarts=40, random_state=0)
print(f"{'kappa':>6} | {'pred mean':>10} | {'on bound':>9} | {'UPPER bound':>22}")
for row in kappa_rows:
    up, lo = bound_report(row["x_next"])
    print(f"{row['kappa']:6g} | {row['pred_mean']:+10.4f} | {len(up) + len(lo):>7}/{D}"
          f" | {str(up) if up else 'none':>22}")

fig = plot_kappa_sensitivity(X, y, bounds, gp_check, kappa_rows, ucb_acquisition, maximize=True)
plt.show()


def restricted_ucb(kappa, n_starts=100):
    """Maximise UCB over the live axes only, holding flat axes at the incumbent."""
    lo = bounds[live, 0]
    hi = bounds[live, 1]

    def neg(v):
        p = incumbent.copy()
        p[live] = v
        return -ucb_acquisition(normalize(p.reshape(1, -1), bounds), gp_check,
                                 kappa=kappa, maximize=True)[0]

    best_val, best_v = np.inf, None
    rng = np.random.default_rng(0)
    for start in rng.uniform(lo, hi, size=(n_starts, len(live))):
        res = minimize(neg, start, method="L-BFGS-B", bounds=list(zip(lo, hi)))
        if res.fun < best_val:
            best_val, best_v = res.fun, res.x
    p = incumbent.copy()
    p[live] = best_v
    m, s = gp_check.predict(normalize(p.reshape(1, -1), bounds), return_std=True)
    return p, m[0], s[0]


print(f"\n=== restricted to the live axes x{live}, holding x{flat} at the incumbent ===")
print(f"{'kappa':>6} | {'pred mean':>10} | {'pred std':>9} | {'UPPER bnd':>12}"
      f" | {'lower bnd (ok)':>18} | {'dist':>7}")
for k in [0.0, 0.25, 0.5, 1.0]:
    p, m, s = restricted_ucb(k)
    up, lo_ = bound_report(p)
    print(f"{k:6g} | {m:+10.4f} | {s:9.4f} | {str(up) if up else 'none':>12}"
          f" | {str(lo_) if lo_ else 'none':>18} | {np.linalg.norm(p - incumbent):7.4f}")
print(f"incumbent y = {y.max():.6f}")

# Committed choice -- reused by the proposal, the slice plot, and the diagnostics
# replay below, so they can't silently drift apart.
KAPPA = 0.25
print(f"\nusing kappa = {KAPPA}, searching only x{live}")
print("chosen as the largest kappa touching no UPPER bound while keeping a hedge")

## Propose the next point

The live axes come from the restricted UCB search; x4 and x7 are held at the incumbent because the acquisition is effectively flat along them and letting it choose them pushed x4 onto its upper bound.

The convex-hull check is omitted here -- at D=8 with 41 points every observation is a hull vertex, so the test carries no information (see the dataset cell). The per-axis range check and the upper/lower bound split are what matter.

In [ ]:
x_next, mu_next, sigma_next = restricted_ucb(KAPPA, n_starts=200)
gp = gp_check

print(f"--- Next point to evaluate (bounds shape {bounds.shape}) ---")
print("x_next:", np.round(x_next, 6))
print(gp.kernel_)
print(f"GP predicted mean: {mu_next:.6f}, predicted std: {sigma_next:.6f}")
print(f"current best observed y: {y.max():.6f}"
      f"  -> predicted improvement: {mu_next - y.max():+.6f}")
print(f"flat axes x{flat} held at the incumbent:"
      f" {np.allclose(x_next[flat], incumbent[flat])}")

up, lo = bound_report(x_next)
print("\non UPPER bound (pad_fraction artifact -- bad):", up or "none")
if up:
    print("*** pad_fraction is choosing those coordinates, not the model.")
    print("    Lower kappa until the proposal comes off the upper bounds. ***")
print("on lower bound (X >= 0 floor -- legitimate):", lo or "none")
if lo:
    print("  corroborating rank correlations for those axes:",
          {f"x{d}": round(float(srs[d]), 3) for d in lo})

outside = [d for d in range(D) if not (X[:, d].min() <= x_next[d] <= X[:, d].max())]
print("outside the observed range on its own axis:", outside or "none")

print(f"\ndistance from the incumbent: {np.linalg.norm(x_next - incumbent):.6f}")
print("nearest 3 observations to x_next:")
for i in np.argsort(np.linalg.norm(X - x_next, axis=1))[:3]:
    print(f"  dist={np.linalg.norm(X[i] - x_next):.4f}  y={y[i]:+.6f}")

## Visualise the GP and acquisition function via 1D slices

Eight panels, each holding the other seven dimensions fixed at the current best observed point and sweeping one dimension. Dotted line is the fixed centre, dashed red is the proposed `x_next`, green is UCB.

Expect **x7 to be a flat line and x4 nearly flat** -- their y-axis ranges are about 0.08 and 0.86 against x2's 7.3. That is why those two coordinates are pinned at the incumbent rather than optimised. x0, x2 and x6 should show the most structure, matching both the length-scales and the rank correlations.

This is a *partial* view: eight 1D slices through one point say nothing about interactions between dimensions, and at D=8 that omission covers most of the space. Treat it as a sanity check on the marginal behaviour, not a picture of the surface.

In [ ]:
fig = plot_nd_slices(
    X, y, bounds, gp,
    acquisition_fn=ucb_acquisition,
    x_next=x_next,
    acq_kwargs={"kappa": KAPPA, "maximize": True},
)
plt.show()

## Sanity-check the surrogate model: leave-one-out calibration

Refits the GP once per observation, leaving it out, and predicts it from the rest. At D=8 this is essentially the only way to judge the surrogate -- the slice plots cover a vanishing fraction of the space.

Expect the best coverage in the set: around 40 of 41 points inside their own 95% interval, with the worst-predicted being an ordinary initial observation missed by a modest margin -- not the newly collected point (predicted to 1.5 sigma before it arrived) and not the incumbent. Contrast function 7, where the incumbent itself is a 13.5 sigma LOO miss.

In [ ]:
pred_mean, pred_std = loo_predictions(X, y, bounds, gp_kwargs={"n_restarts_optimizer": 20})
fig = plot_loo_calibration(y, pred_mean, pred_std)
plt.show()

within = np.abs(y - pred_mean) <= 1.96 * pred_std
print(f"points inside their own 95% LOO interval: {within.sum()}/{len(y)}")
worst = int(np.argmax(np.abs(y - pred_mean)))
print(f"worst-predicted: index {worst} -> true {y[worst]:.6f},"
      f" predicted {pred_mean[worst]:.6f} (std {pred_std[worst]:.6f})"
      f" = {(y[worst] - pred_mean[worst]) / pred_std[worst]:+.1f} sigma")
print("is that the newly collected point?", worst == n_initial)
print("is it the incumbent?", worst == int(np.argmax(y)))

## Iteration diagnostics

`compute_iteration_diagnostics` replays the ordered `X`/`y` to reconstruct what the acquisition value, GP hyperparameters, and domain-wide uncertainty were at each past proposal -- no persisted log involved.

**`domain_grid_n` MUST be lowered drastically at D=8.** Its `domain_mean_std` field averages the posterior std over a `domain_grid_n ** D` grid, so the default `domain_grid_n=40` means `40**8 = 6.6e12` points -- about **419,000 GB**. Even `domain_grid_n=10` would be 1e8 points. The cell below sizes the grid so the point count stays near 200,000, which gives `domain_grid_n=4` here: just four samples per axis.

At that resolution `domain_mean_std` is a very coarse estimate of domain-average uncertainty. It remains usable for comparing one iteration against another *within this notebook*, because every iteration uses the same grid, but its absolute value means little and it is not comparable across notebooks.

Two further caveats. The setting is taken from `KAPPA` so it can't drift from the proposal -- but the recorded observation came from the old week-1 sweep's `pi`/`xi=0.01`, so its replayed acquisition value describes a decision never made that way; treat that column as meaningless until the history is UCB throughout. And it assumes row order is true chronological order.

`plot_bo_diagnostics` is skipped (it hard-codes a 2D scatter panel and this is 8D); the trend plots below are dimension-agnostic and gated on having a few completed iterations.

In [ ]:
# compute_iteration_diagnostics builds a domain_grid_n ** D grid. The default of
# 40 is 40**8 = 6.6e12 points (~419,000 GB) at D=8. Size it so the grid stays
# ~200k points whatever D is -- here that means just 4 samples per axis.
DOMAIN_GRID_N = max(3, int(200_000 ** (1.0 / D)))
print(f"domain_grid_n = {DOMAIN_GRID_N} -> {DOMAIN_GRID_N ** D:,} grid points"
      f" (the default 40 would be {40 ** D:,})")
print("domain_mean_std is therefore very coarse: comparable across iterations")
print("within this notebook, not across notebooks or grid sizes.")

history = None
if len(new_y) == 0:
    print("\nNo observations beyond the initial batch yet -- nothing to replay.")
    print("Best y so far:", y.max())
else:
    history = compute_iteration_diagnostics(X, y, bounds, n_initial=n_initial,
                                             acquisition="ucb", kappa=KAPPA, maximize=True,
                                             domain_grid_n=DOMAIN_GRID_N)
    print("\nBest y so far:", np.nanmax(history["y"]))
    print("completed iterations beyond the initial batch:", len(new_y))

    if len(new_y) >= 3:
        for plot_fn in (plot_convergence, plot_acquisition_decay,
                        plot_uncertainty_shrinkage, plot_step_distance):
            plot_fn(history)
            plt.show()
    else:
        print(f"\nOnly {len(new_y)} completed iteration(s) -- need at least 3 for the trend\n"
              "plots to say anything. Skipping them; the raw fields are below.")

## Raw diagnostic fields

In [ ]:
if history is not None:
    print("y (last 5):", np.round(history["y"][-5:], 5))
    print("iteration (last 5):", history["iteration"][-5:])
    obs = ~np.isnan(history["acq_value"])
    print("\nfor the proposed (non-initial) points only:")
    print("  acq_value      :", history["acq_value"][obs])
    print("  pred_mean      :", history["pred_mean"][obs])
    print("  actual y       :", history["y"][obs])
    print("  length_scale   :", history["length_scale"][obs])
    print("  domain_mean_std:", history["domain_mean_std"][obs])

In [ ]:
# The proposal as a hyphen-separated string, for submission.
print("-".join(f"{v:.6f}" for v in x_next))

# Full precision as well. 6 dp is fine to submit, but paste THIS into next
# week's new_X: a 6-dp copy of function 4's proposal rounded 4e-7 outside its
# own upper bound and tripped validate_bounds_consistency.
print("\nfull precision (use for next week's new_X):")
print("-".join(repr(float(v)) for v in x_next))